In [ ]:
import json
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore")

import utils
import utils_stage

# ── Palette ──
C_BLUE = "#3498db"
C_RED = "#e74c3c"
C_GREEN = "#57bb8a"
C_PURPLE = "#9b59b6"
C_ORANGE = "#e67e22"
C_TEAL = "#1abc9c"
C_DARK = "#2c3e50"

JUDGE_PATH = Path("data") / "llm_judge_style_results.json"
CACHE_PATH = Path("data") / "all_sessions.parquet"

# ── Load data ──
df_raw = pd.read_parquet(CACHE_PATH)
df_raw = df_raw.drop_duplicates(subset="session_id")
df_raw["session_start"] = pd.to_datetime(df_raw["session_start"], utc=True)
df_raw["session_end"] = pd.to_datetime(df_raw["session_end"], utc=True)
print(f"Loaded {len(df_raw)} sessions")

# ── Process ──
df = utils.process_triage_data(df_raw)
stage_metrics = df["chat_history"].apply(utils_stage.analyze_session_stages)
df = pd.concat([df, stage_metrics], axis=1)

# ── Parse messages ──
_MSG_PATTERN = re.compile(r'(\d{4}-\d{2}-\d{2}[T ]\S+)\|\|\|\[(\w+)\]:\s*')

def parse_patient_messages(chat_history_str):
    chat_text = str(chat_history_str)
    matches = list(_MSG_PATTERN.finditer(chat_text))
    messages = []
    for i, m in enumerate(matches):
        sender = m.group(2).lower()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(chat_text)
        text = chat_text[start:end].strip()
        if sender == "patient" and text:
            messages.append(text)
    return messages

msg_rows = []
conv_rows = []
for _, row in df.iterrows():
    patient_msgs = parse_patient_messages(row["chat_history"])
    for m in patient_msgs:
        words = m.split()
        msg_rows.append({
            "session_id": row["session_id"],
            "interaction_type": row["interaction_type"],
            "text": m,
            "word_count": len(words),
            "has_punctuation": bool(re.search(r"[.!?,;:]", m)),
            "all_lowercase": m == m.lower(),
            "all_uppercase": m == m.upper() and len(m) > 2,
            "starts_lowercase": m[0].islower() if m else False,
            "single_word": len(words) == 1,
            "numeric_only": bool(re.match(r"^[\d.,%/\s-]+$", m.strip())),
            "is_question": "?" in m,
        })
    word_counts = [len(m.split()) for m in patient_msgs] if patient_msgs else [0]
    conv_rows.append({
        "session_id": row["session_id"],
        "interaction_type": row["interaction_type"],
        "final_outcome": row["final_outcome"],
        "major_group": row["major_group"],
        "n_patient_turns": len(patient_msgs),
        "total_patient_words": sum(word_counts),
    })

df_msgs = pd.DataFrame(msg_rows)
df_conv = pd.DataFrame(conv_rows)

# ── Judge ──
has_judge = JUDGE_PATH.exists()
if has_judge:
    with open(JUDGE_PATH) as f:
        judge_data = json.load(f)
    df_judge = pd.DataFrame(judge_data["results"])
    print(f"Judge: {len(df_judge)} conversations, model={judge_data['model']}")
else:
    df_judge = None
    print("No judge results found")

print(f"Conversations: {len(df_conv)}, Messages: {len(df_msgs)}")

In [ ]:
# ── Figure 2: 2×3 combined panel ──

# --- Setup ---
completion_map = {
    "Regular Completed Sessions": "Complete",
    "Early Cutoff Sessions": "Early cutoff",
    "Terminated: Abandoned Mid-Conversation": "Incomplete",
    "Terminated: Unhandled Symptoms": "Incomplete",
    "Terminated: Ineligible": "Incomplete",
    "Other Terminations": "Incomplete",
}
df_conv["completion"] = df_conv["major_group"].map(completion_map).fillna("Incomplete")

MIN_TURNS = 2
df_conv_included = df_conv[df_conv["n_patient_turns"] >= MIN_TURNS]
included_ids = set(df_conv_included["session_id"])
df_msgs_included = df_msgs[df_msgs["session_id"].isin(included_ids)]

N_total = len(df_conv)
N_included = len(df_conv_included)
N_msgs = len(df_msgs_included)

if has_judge:
    df_j = df_judge[df_judge["session_id"].isin(included_ids)].copy()
    N_judge = len(df_j)
else:
    df_j = None
    N_judge = 0

C_COMP = "#57bb8a"
C_EARLY = "#e67e22"
C_INC = "#95a5a6"
C_CUTOFF = "#e74c3c"

from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.patches import Rectangle

comma_fmt = FuncFormatter(lambda x, _: f"{int(x):,}")

fig = plt.figure(figsize=(18, 11))
gs = GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.45,
             left=0.07, right=0.97, top=0.93, bottom=0.05)

gs_a = gs[0, 0].subgridspec(3, 1, hspace=0.45)
ax_a = [fig.add_subplot(gs_a[i]) for i in range(3)]

ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[1, 0])
ax_e = fig.add_subplot(gs[1, 1])
ax_f = fig.add_subplot(gs[1, 2])

# ═══ (a) Patient turns — 3 stacked KDE ═══
max_t = 35
x_grid = np.linspace(0, max_t, 500)

kde_groups = [
    ("Incomplete session (user abandoned)", C_INC, ax_a[0]),
    ("Early emergency flag", C_EARLY, ax_a[1]),
    ("Completed session", C_COMP, ax_a[2]),
]

for label, color, ax in kde_groups:
    comp_key = {"Incomplete session (user abandoned)": "Incomplete",
                "Early emergency flag": "Early cutoff",
                "Completed session": "Complete"}[label]
    data = df_conv[df_conv["completion"] == comp_key]["n_patient_turns"].values.astype(float)
    n = len(data)
    if n < 2:
        continue
    kde = gaussian_kde(data, bw_method=0.5)
    y = np.maximum(kde(x_grid) * n, 0)

    ax.fill_between(x_grid, y, alpha=0.6, color=color, zorder=2)
    ax.plot(x_grid, y, color=color, linewidth=1, zorder=3)
    ax.axvline(MIN_TURNS, color=C_CUTOFF, linestyle="--", linewidth=1.5, zorder=6)
    ax.axvspan(0, MIN_TURNS, color=C_CUTOFF, alpha=0.06, zorder=0)
    ax.set_xlim(0, max_t)
    ax.set_ylim(0, y.max() * 1.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(f"{label} (n={n:,})", fontsize=9, loc="left", pad=4)
    ax.set_ylabel("Density", fontsize=8)
    ax.tick_params(axis="y", labelsize=7)
    ax.yaxis.set_major_formatter(comma_fmt)

    med = np.median(data)
    ax.text(0.95, 0.5, f"median = {med:.0f}",
            transform=ax.transAxes, fontsize=7, color="0.3",
            ha="right", va="center")

    if ax is not ax_a[-1]:
        ax.tick_params(axis="x", labelbottom=False)
    else:
        ax.set_xlabel("Patient turns", fontsize=10)

yl_top = ax_a[0].get_ylim()
ax_a[0].text(MIN_TURNS + 0.4, yl_top[1] * 0.85,
             f"$\geq${MIN_TURNS} turns cutoff (filtered for subsequent analysis)",
             fontsize=7, color=C_CUTOFF, va="top")

bbox_a0 = ax_a[0].get_position()
fig.text(bbox_a0.x0, bbox_a0.y1 + 0.02,
         f"(a) Patient turns per conversation (n={N_total:,} conversations)",
         fontsize=11, ha="left", va="bottom")

# ═══ (b) Words per message ═══
ax = ax_b
wc = df_msgs_included["word_count"].values
wc_clamped = wc[wc <= 40]
N_clamped = len(wc_clamped)
N_excluded = len(wc) - N_clamped
pct_excluded = 100 * N_excluded / len(wc)

bins_w = np.arange(0.5, 41.5, 1)
ax.hist(wc_clamped, bins=bins_w, color=C_BLUE, edgecolor="white",
        linewidth=0.3, alpha=0.85, zorder=3)

ax.set_xlabel("Words per message", fontsize=10)
ax.set_ylabel("Messages", fontsize=10)
ax.set_title(f"(b) Words (n={N_clamped:,} messages, n={N_included:,} conversations)",
             fontsize=11, loc="left", pad=12)
ax.set_xlim(0, 41)
ax.set_ylim(0, 1500)
ax.yaxis.set_major_locator(MultipleLocator(500))
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.set_major_formatter(comma_fmt)

ax.text(0.16, 0.99, "y-axis truncated at 1,500",
        transform=ax.transAxes, fontsize=7, color="0.4",
        ha="left", va="top")

zoom_rect = Rectangle((0.5, 0), 5, 1500, linewidth=1.2,
                       edgecolor="0.7", facecolor="0.92", alpha=0.5, zorder=1)
ax.add_patch(zoom_rect)

inset = ax.inset_axes([0.40, 0.38, 0.56, 0.35])
bins_zoom = np.arange(0.5, 6.5, 1)
counts_zoom, _, patches_zoom = inset.hist(wc_clamped, bins=bins_zoom,
                                           color=C_BLUE, edgecolor="white",
                                           linewidth=0.5, alpha=0.85)
for i, (patch, cnt) in enumerate(zip(patches_zoom, counts_zoom)):
    pct = 100 * cnt / N_clamped
    inset.text(i + 1, cnt + N_clamped * 0.003, f"{int(cnt):,}\n({pct:.0f}%)",
               ha="center", va="bottom", fontsize=6.5, color="0.3")
inset.set_xlim(0.3, 5.7)
inset.set_ylim(0, max(counts_zoom) * 1.4)
inset.set_xticks([1, 2, 3, 4, 5])
inset.set_xlabel("Words per message", fontsize=7)
inset.tick_params(axis="y", labelsize=7)
inset.tick_params(axis="x", labelsize=7)
inset.yaxis.set_major_formatter(comma_fmt)
inset.spines[["top", "right"]].set_visible(False)
inset.patch.set_alpha(1.0)
inset.patch.set_facecolor("#f5f5f5")
for spine in inset.spines.values():
    spine.set_edgecolor("0.7")
    spine.set_linewidth(1.2)

n_under6 = int((df_msgs_included["word_count"] <= 5).sum())
pct_under6 = 100 * n_under6 / len(df_msgs_included)
inset.set_title(f"{n_under6:,} messages ({pct_under6:.1f}%) with <6 words",
                fontsize=7, color="0.3", pad=3)

if N_excluded > 0:
    ax.text(0.97, 0.05,
            f"{N_excluded:,} messages ({pct_excluded:.1f}%)\nwith >40 words",
            transform=ax.transAxes, fontsize=7, color="0.4",
            ha="right", va="bottom")

# ═══ (c) Symptom category ═══
ax = ax_c

if df_j is not None and "symptom_category" in df_j.columns:
    cat_counts = df_j["symptom_category"].value_counts()
    grey_cats = {"unrelated", "other_mixed", "unevaluable"}
    clinical = [(k, v) for k, v in cat_counts.items() if k not in grey_cats]
    non_clinical = [(k, v) for k, v in cat_counts.items() if k in grey_cats]
    ordered = clinical + non_clinical

    label_map = {
        "respiratory": "Respiratory", "cardiovascular": "Cardiovascular",
        "gastrointestinal": "Gastrointestinal", "genitourinary": "Genitourinary",
        "musculoskeletal": "Musculoskeletal", "neurological": "Neurological",
        "constitutional": "Constitutional", "dermatological": "Dermatological",
        "mental_health": "Mental health", "ent": "ENT", "endocrine": "Endocrine",
        "other_mixed": "Other / mixed", "unrelated": "Unrelated",
        "unevaluable": "Unevaluable",
    }
    cat_labels = [label_map.get(k, k) for k, _ in ordered]
    cat_vals = [v for _, v in ordered]
    cat_colors = [C_TEAL if k not in grey_cats else "#bdc3c7" for k, _ in ordered]

    cat_y = np.arange(len(cat_labels))
    ax.barh(cat_y, cat_vals, color=cat_colors, edgecolor="none",
            height=0.65, alpha=0.85, zorder=3)
    for yp, val in zip(cat_y, cat_vals):
        ax.text(val + max(cat_vals) * 0.02, yp,
                f"{val:,} ({100*val/N_judge:.0f}%)", va="center", fontsize=8, color="0.3")
    ax.set_yticks(cat_y)
    ax.set_yticklabels(cat_labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, max(cat_vals) * 1.35)
    ax.xaxis.set_major_formatter(comma_fmt)

ax.set_xlabel("Conversations", fontsize=10)
ax.set_title(f"(c) Symptom category (n={N_judge:,} conversations)",
             fontsize=11, loc="left", pad=12)
ax.spines[["top", "right"]].set_visible(False)

# ═══ (d) Information shared ═══
ax = ax_d

if df_j is not None:
    rows_d = []

    df_j["health_literacy"] = pd.to_numeric(df_j["health_literacy"], errors="coerce")
    hl_blues = ["#a8d8ea", "#5dade2", "#2471a3"]
    for lvl, lbl in [(1, "Basic"), (2, "Medium"), (3, "Critical")]:
        rows_d.append(("Health\nliteracy", lbl,
                        int((df_j["health_literacy"] == lvl).sum()), hl_blues[lvl - 1]))

    recall_blues = {"low": "#aed6f1", "medium": "#5499c7", "high": "#1a5276"}
    for lvl in ["low", "medium", "high"]:
        rows_d.append(("Recall", lvl.capitalize(),
                        int((df_j["recall"] == lvl).sum()), recall_blues[lvl]))

    rows_d.append(("Sensitive\ntopic", "Yes",
                    int(df_j["sensitive_topic"].sum()), "#1e8449"))
    rows_d.append(("Sensitive\ntopic", "No",
                    int((~df_j["sensitive_topic"]).sum()), "#c0392b"))

    if "oversharing" in df_j.columns:
        rows_d.append(("Oversharing", "Yes",
                        int(df_j["oversharing"].sum()), "#1e8449"))
        rows_d.append(("Oversharing", "No",
                        int((~df_j["oversharing"]).sum()), "#c0392b"))

    bar_vals_d = [r[2] for r in rows_d]
    bar_colors_d = [r[3] for r in rows_d]
    bar_labels_d = [r[1] for r in rows_d]

    y_positions = []
    group_spans = {}
    prev_group = None
    y = 0
    for group, sublabel, val, color in rows_d:
        if group != prev_group and prev_group is not None:
            y += 1.4
        y_positions.append(y)
        if group not in group_spans:
            group_spans[group] = [y, y]
        else:
            group_spans[group][1] = y
        prev_group = group
        y += 1
    y_positions = np.array(y_positions)

    ax.barh(y_positions, bar_vals_d, color=bar_colors_d, edgecolor="none",
            height=0.85, alpha=0.85, zorder=3)
    for yp, val in zip(y_positions, bar_vals_d):
        ax.text(val + max(bar_vals_d) * 0.02, yp,
                f"{val:,} ({100*val/N_judge:.0f}%)", va="center", fontsize=8, color="0.3")

    ax.set_yticks(y_positions)
    ax.set_yticklabels(bar_labels_d, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, max(bar_vals_d) * 1.35)
    ax.set_ylim(y_positions[-1] + 0.7, y_positions[0] - 1.5)
    ax.xaxis.set_major_formatter(comma_fmt)

    for gname, (ystart, yend) in group_spans.items():
        mid_y = (ystart + yend) / 2
        ax.annotate(gname, xy=(0, mid_y),
                    xycoords=ax.get_yaxis_transform(),
                    xytext=(-62, 0), textcoords="offset points",
                    ha="center", va="center", fontsize=9,
                    color="0.2", rotation=90, annotation_clip=False)

ax.set_xlabel("Conversations", fontsize=10)
ax.set_title(f"(d) Information shared (n={N_judge:,} conversations)",
             fontsize=11, loc="left", pad=12)
ax.spines[["top", "right"]].set_visible(False)

# ═══ (e) Emotional signals ═══
ax = ax_e

if df_j is not None:
    feeling_cols = {
        "feeling_anxious": "Anxiousness",
        "feeling_frustrated": "Frustration",
        "feeling_impatient": "Impatience",
        "feeling_sceptical": "Scepticism",
        "feeling_embarrassed": "Embarrassment",
    }
    f_labels = list(feeling_cols.values())
    f_true = [int(df_j[c].sum()) for c in feeling_cols]

    has_any_emotion = df_j[list(feeling_cols.keys())].any(axis=1)
    n_at_least_one = int(has_any_emotion.sum())
    n_none = int((~has_any_emotion).sum())

    f_labels.extend(["At least one", "None"])
    f_true.extend([n_at_least_one, n_none])
    f_colors = [C_BLUE] * 5 + ["#1e8449", "#c0392b"]

    n_blue = 5
    y_pos = list(range(n_blue))
    y_pos.append(n_blue + 0.5)
    y_pos.append(n_blue + 1.2)
    y_pos = np.array(y_pos)

    ax.barh(y_pos, f_true, color=f_colors, edgecolor="none",
            height=0.6, alpha=0.85, zorder=3)
    for yp, val in zip(y_pos, f_true):
        ax.text(val + max(f_true) * 0.02, yp,
                f"{val:,} ({100*val/N_judge:.0f}%)", va="center", fontsize=8, color="0.3")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(f_labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, max(f_true) * 1.35)
    ax.xaxis.set_major_formatter(comma_fmt)

ax.set_xlabel("Conversations", fontsize=10)
ax.set_title(f"(e) Emotional signals (n={N_judge:,} conversations)",
             fontsize=11, loc="left", pad=12)
ax.spines[["top", "right"]].set_visible(False)

# ═══ (f) Communication style ═══
ax = ax_f

if df_j is not None:
    comm_items = []
    if "non_standard_capitalisation" in df_j.columns:
        comm_items.append(("Non-standard capitalisation",
                           int(df_j["non_standard_capitalisation"].sum()),
                           int(df_j["non_standard_capitalisation"].notna().sum())))
        comm_items.append(("Non-standard punctuation",
                           int(df_j["non_standard_punctuation"].sum()),
                           int(df_j["non_standard_punctuation"].notna().sum())))
    elif "non_standard_style" in df_j.columns:
        comm_items.append(("Non-standard capitalisation\nand punctuation",
                           int(df_j["non_standard_style"].sum()),
                           int(df_j["non_standard_style"].notna().sum())))
    comm_items.append(("Grammar issues",
                       int(df_j["grammar_issues"].sum()),
                       int(df_j["grammar_issues"].notna().sum())))
    comm_items.append(("Typos",
                       int(df_j["typos"].sum()),
                       int(df_j["typos"].notna().sum())))

    comm_bool_cols = []
    if "non_standard_capitalisation" in df_j.columns:
        comm_bool_cols += ["non_standard_capitalisation", "non_standard_punctuation"]
    elif "non_standard_style" in df_j.columns:
        comm_bool_cols += ["non_standard_style"]
    comm_bool_cols += ["grammar_issues", "typos"]
    has_any_comm = df_j[comm_bool_cols].any(axis=1)
    n_at_least_one_comm = int(has_any_comm.sum())
    n_none_comm = int((~has_any_comm).sum())

    c_labels = [x[0] for x in comm_items] + ["At least one", "None"]
    c_vals = [x[1] for x in comm_items] + [n_at_least_one_comm, n_none_comm]
    c_totals = [x[2] for x in comm_items] + [N_judge, N_judge]
    c_colors = [C_BLUE] * len(comm_items) + ["#1e8449", "#c0392b"]

    n_blue_f = len(comm_items)
    y_pos = list(range(n_blue_f))
    y_pos.append(n_blue_f + 0.5)
    y_pos.append(n_blue_f + 1.2)
    y_pos = np.array(y_pos)

    ax.barh(y_pos, c_vals, color=c_colors, edgecolor="none",
            height=0.6, alpha=0.85, zorder=3)
    for yp, val, total in zip(y_pos, c_vals, c_totals):
        pct = 100 * val / total if total else 0
        ax.text(val + max(c_vals) * 0.02, yp,
                f"{val:,} ({pct:.0f}%)", va="center", fontsize=8, color="0.3")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(c_labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, max(c_vals) * 1.35)
    ax.xaxis.set_major_formatter(comma_fmt)

ax.set_xlabel("Conversations", fontsize=10)
ax.set_title(f"(f) Communication style (n={N_judge:,} conversations)",
             fontsize=11, loc="left", pad=12)
ax.spines[["top", "right"]].set_visible(False)

plt.savefig("figure2.pdf", bbox_inches="tight", dpi=300)
plt.savefig("figure2.png", bbox_inches="tight", dpi=300)
plt.show()

print(f"\nFigure 2 saved. n={N_included:,} conversations (>={MIN_TURNS} patient turns) from {N_total:,} total.")
print(f"Judge: {N_judge:,} conversations. Messages (<=40 words): {N_clamped:,}.")